# 04 — End-to-End Backtest Demo

Full pipeline walkthrough:
1. **Data** — Initialize Qlib & load CSI300
2. **Factors** — Create Alpha158 dataset
3. **Model** — Train LightGBM model
4. **Backtest** — Run backtest with TopkDropout strategy
5. **Analysis** — Performance metrics & visualizations

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120

print(f"Project root: {PROJECT_ROOT}")

## Step 1: Initialize Qlib

In [ ]:
from big_a.qlib_config import init_qlib
from qlib.data import D

init_qlib()

instruments = list(D.instruments("csi300"))
print(f"CSI300 instruments: {len(instruments)}")

## Step 2: Create Alpha158 Dataset

In [ ]:
from big_a.factors.handler import create_alpha158_dataset, get_segment_data

dataset = create_alpha158_dataset()

train_features = get_segment_data(dataset, segment="train", col_set="feature")
test_features = get_segment_data(dataset, segment="test", col_set="feature")
test_labels = get_segment_data(dataset, segment="test", col_set="label")

print(f"Train features: {train_features.shape}")
print(f"Test features:  {test_features.shape}")
print(f"Test labels:    {test_labels.shape}")

## Step 3: Train LightGBM Model

In [ ]:
from big_a.models.lightgbm_model import train, predict_to_dataframe, save_model

model, dataset_out, config = train()

# Generate test predictions
preds = predict_to_dataframe(model, dataset_out, segment="test")
print(f"Test predictions: {preds.shape}")
print(f"Score stats:\n{preds['score'].describe()}")

## Step 4: Run Backtest

In [ ]:
from big_a.backtest.engine import run_backtest, compute_analysis

report, positions = run_backtest(preds)

print(f"Backtest report: {len(report)} trading days")
print(f"Date range: {report.index[0]} → {report.index[-1]}")
print(f"\nReport columns: {list(report.columns)}")
report.head()

## Step 5: Performance Analysis

In [ ]:
from big_a.backtest.analysis import analyze_backtest, generate_report

analysis = analyze_backtest(report)

print("=" * 60)
print("  BACKTEST PERFORMANCE SUMMARY")
print("=" * 60)
print(f"  Period: {analysis['start_date']} → {analysis['end_date']} ({analysis['n_trading_days']} days)")
print(f"  Annualized Return:    {analysis['annualized_return']:>+9.2%}")
print(f"  Annualized Benchmark: {analysis['annualized_benchmark']:>+9.2%}")
print(f"  Excess Return:        {analysis['excess_return']:>+9.2%}")
print(f"  Sharpe Ratio:         {analysis['sharpe_ratio']:>+9.4f}")
print(f"  Max Drawdown:         {analysis['max_drawdown']:>9.2%}")
print(f"  Information Ratio:    {analysis['information_ratio']:>+9.4f}")
print(f"  Total Cost:           {analysis['total_cost']:>9.4f}")
print(f"  Mean Turnover:        {analysis['mean_turnover']:>9.4f}")
print("=" * 60)

In [ ]:
# NAV Curve
cum_strategy = (1 + report["return"]).cumprod()
cum_bench = (1 + report["bench"]).cumprod()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(cum_strategy.index, cum_strategy.values, label="Strategy", linewidth=1.2)
ax.plot(cum_bench.index, cum_bench.values, label="Benchmark", linewidth=1.0, alpha=0.7)
ax.set_title("NAV Curve: Strategy vs Benchmark")
ax.set_ylabel("NAV")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown Chart
peak = cum_strategy.cummax()
drawdown = (cum_strategy - peak) / peak

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(drawdown.index, drawdown.values, 0, color="red", alpha=0.4)
ax.plot(drawdown.index, drawdown.values, color="darkred", linewidth=0.6)
ax.set_title("Drawdown")
ax.set_ylabel("Drawdown")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly Returns Heatmap
monthly = report["return"].resample("ME").apply(lambda x: (1 + x).prod() - 1)
years = monthly.index.year.unique()

data = np.full((len(years), 12), np.nan)
for i, year in enumerate(years):
    for j, month in enumerate(range(1, 13)):
        mask = (monthly.index.year == year) & (monthly.index.month == month)
        if mask.any():
            data[i, j] = monthly.loc[mask].iloc[-1]

fig, ax = plt.subplots(figsize=(10, max(3, len(years) * 0.6)))
im = ax.imshow(data, aspect="auto", cmap="RdYlGn", vmin=-0.1, vmax=0.1)
ax.set_xticks(range(12))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_yticks(range(len(years)))
ax.set_yticklabels([str(y) for y in years])
ax.set_title("Monthly Returns")

for i in range(len(years)):
    for j in range(12):
        if not np.isnan(data[i, j]):
            ax.text(j, i, f"{data[i, j]:.1%}", ha="center", va="center", fontsize=7)

plt.colorbar(im, ax=ax, label="Return")
plt.tight_layout()
plt.show()

In [ ]:
# Save report to output directory
output_dir = PROJECT_ROOT / "output" / "backtest_demo"
analysis_copy = {**analysis, "_report_df": report}
report_path = generate_report(analysis_copy, output_dir)
print(f"Report saved to: {report_path}")